# Comparison to SQD with N$_2$

## Setup

In [ ]:
#pip install qiskit qiskit

In [30]:
import warnings

warnings.filterwarnings("ignore")

import pyscf
import pyscf.cc
import pyscf.mcscf

# Specify molecule properties
open_shell = False
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()

geometry = [["N", (0, 0, 0)], ["N", (1.0, 0, 0)]]
mol.build(
    atom=geometry,
    basis="6-31g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2

cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

# Compute exact energy
exact_energy = cas.run().e_tot

converged SCF energy = -108.835236570774
CASCI E = -109.046671778080  E(CI) = -32.8155692383188  S^2 = 0.0000000


## Convert to/construct OpenFermion Hamiltonian

In [56]:
import openfermion as of

In [61]:
charge = mol.charge
multiplicity = mol.multiplicity
basis = mol.basis

In [76]:
# geometry = geometry_from_pubchem("n2")

ofham = MolecularData(geometry, basis, charge=0, multiplicity=1)
ofmol = run_pyscf(ofham, run_mp2=True, run_cisd=False, run_ccsd=False, run_fci=False)

ham = MolecularData(filename=ofmol.filename)
ham = ham.get_molecular_hamiltonian()
ham = of.get_fermion_operator(ham)
ham = of.transforms.freeze_orbitals(ham, occupied=[0, 1, 2, 3])
ham = of.jordan_wigner(ham)

In [86]:
nqubits = of.utils.count_qubits(ham)
nterms = len(ham.terms)

In [87]:
print(f"Qubit Hamiltonian acts on {nqubits} qubit(s) and has {nterms} term(s).")

Qubit Hamiltonian acts on 32 qubit(s) and has 21521 term(s).


## Sorted insertion with $k$-commuting shot counts

In [90]:
import cirq
import quimb.tensor as qtn

from kcommute import get_si_sets, r_hat_measurement_count
from kcommute.commute import compute_blocks
from kcommute.shot_metrics import get_shotcounts_mpo_mps
from kcommute.tensor_nets import groups_of_to_mpos

In [91]:
qubits = cirq.LineQubit.range(nqubits)

In [89]:
k = 1

In [ ]:
groups = get_si_sets(ham, compute_blocks(qubits, k))

## Ansatz

In [2]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -109.0398256929733  E_corr = -0.2045891221988309


In [4]:
import ffsim
from qiskit import QuantumCircuit, QuantumRegister

n_reps = 1
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices = [(p, p) for p in range(0, num_orbitals, 4)]

ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices),
)

nelec = (num_elec_a, num_elec_b)

# create an empty quantum circuit
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

In [5]:
circuit.draw(fold=-1)

┌───────────────────┐┌───────────────────┐ ░ ┌─┐                                                                                             
    q_0: ┤0                  ├┤0                  ├─░─┤M├─────────────────────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░ └╥┘┌─┐                                                                                          
    q_1: ┤1                  ├┤1                  ├─░──╫─┤M├──────────────────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║ └╥┘┌─┐                                                                                       
    q_2: ┤2                  ├┤2                  ├─░──╫──╫─┤M├───────────────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║ └╥┘┌─┐                                                                                    
    q_3: ┤3                  ├┤3                  ├─░──╫──╫──╫─┤M├────────────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║ └╥┘┌─┐                                                                                 
    q_4: ┤4                  ├┤4                  ├─░──╫──╫──╫──╫─┤M├─────────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║ └╥┘┌─┐                                                                              
    q_5: ┤5                  ├┤5                  ├─░──╫──╫──╫──╫──╫─┤M├──────────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║ └╥┘┌─┐                                                                           
    q_6: ┤6                  ├┤6                  ├─░──╫──╫──╫──╫──╫──╫─┤M├───────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                                        
    q_7: ┤7                  ├┤7                  ├─░──╫──╫──╫──╫──╫──╫──╫─┤M├────────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                                     
    q_8: ┤8                  ├┤8                  ├─░──╫──╫──╫──╫──╫──╫──╫──╫─┤M├─────────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                                  
    q_9: ┤9                  ├┤9                  ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├──────────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                               
   q_10: ┤10                 ├┤10                 ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                            
   q_11: ┤11                 ├┤11                 ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├────────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                         
   q_12: ┤12                 ├┤12                 ├─░──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├─────────────────────────────────────────────────────────
         │                   ││                   │ ░  ║  ║  ║  ║  ║  ║  ║  ║  ║  ║

In [6]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

In [7]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

spin_a_layout = [0, 14, 18, 19, 20, 33, 39, 40, 41, 53, 60, 61, 62, 72, 81, 82]
spin_b_layout = [2, 3, 4, 15, 22, 23, 24, 34, 43, 44, 45, 54, 64, 65, 66, 73]
initial_layout = spin_a_layout + spin_b_layout

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, initial_layout=initial_layout
)

# without PRE_INIT passes
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/o pre-init passes): {isa_circuit.count_ops()}")

# with PRE_INIT passes
# We will use the circuit generated by this pass manager for hardware execution
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts (w/ pre-init passes): {isa_circuit.count_ops()}")

Gate counts (w/o pre-init passes): OrderedDict([('rz', 4468), ('sx', 3430), ('ecr', 1366), ('x', 227), ('measure', 32), ('barrier', 1)])
Gate counts (w/ pre-init passes): OrderedDict([('rz', 2486), ('sx', 2149), ('ecr', 730), ('x', 73), ('measure', 32), ('barrier', 1)])
